In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Define paths
raw_path = Path("d:/bluestock_mf_capstone/data/raw")
processed_path = Path("d:/bluestock_mf_capstone/data/processed")
db_path = Path("d:/bluestock_mf_capstone/data/db")

# Create directories if not exist
processed_path.mkdir(parents=True, exist_ok=True)
db_path.mkdir(parents=True, exist_ok=True)

print("Paths set:")
print(f"Raw: {raw_path.absolute()}")
print(f"Processed: {processed_path.absolute()}")
print(f"DB: {db_path.absolute()}")

Paths set:
Raw: d:\bluestock_mf_capstone\data\raw
Processed: d:\bluestock_mf_capstone\data\processed
DB: d:\bluestock_mf_capstone\data\db


In [2]:
# Load all raw CSV files into a dictionary
raw_files = {
    "fund_master": "01_fund_master.csv",
    "nav_history": "02_nav_history.csv",
    "aum_by_fund_house": "03_aum_by_fund_house.csv",
    "monthly_sip_inflows": "04_monthly_sip_inflows.csv",
    "category_inflows": "05_category_inflows.csv",
    "industry_folio_count": "06_industry_folio_count.csv",
    "scheme_performance": "07_scheme_performance.csv",
    "investor_transactions": "08_investor_transactions.csv",
    "portfolio_holdings": "09_portfolio_holdings.csv",
    "benchmark_indices": "10_benchmark_indices.csv"
}

data = {}

for name, fname in raw_files.items():
    path = raw_path / fname
    if path.suffix == '.csv':
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)
    data[name] = df
    print(f"Loaded {name}: {df.shape}")

Loaded fund_master: (40, 15)
Loaded nav_history: (46000, 3)
Loaded aum_by_fund_house: (90, 5)
Loaded monthly_sip_inflows: (48, 6)
Loaded category_inflows: (144, 3)
Loaded industry_folio_count: (21, 6)
Loaded scheme_performance: (40, 19)
Loaded investor_transactions: (32778, 13)
Loaded portfolio_holdings: (322, 8)
Loaded benchmark_indices: (8050, 3)


In [3]:
nav = data['nav_history'].copy()
print(f"Original shape: {nav.shape}")
print(nav.head(2))

# --- Cleaning steps ---
# 1. Parse date column (identify correct column)
date_col = None
for col in nav.columns:
    if 'date' in col.lower():
        date_col = col
        break
print(f"Date column: {date_col}")

# Convert to datetime
nav[date_col] = pd.to_datetime(nav[date_col], errors='coerce')

# 2. Find scheme code column
code_col = None
for col in nav.columns:
    if 'code' in col.lower():
        code_col = col
        break

# 3. Remove duplicates (scheme_code + date)
nav = nav.drop_duplicates(subset=[code_col, date_col])

# 4. Sort by code and date
nav = nav.sort_values([code_col, date_col])

# 5. Forward-fill missing NAV for each scheme (handle weekends/holidays)
nav_col = None
for col in nav.columns:
    if col.lower() in ['nav', 'net_asset_value', 'price']:
        nav_col = col
        break

# Group by scheme code and ffill NAV
nav[nav_col] = nav.groupby(code_col)[nav_col].ffill()

# 6. Validate NAV > 0
invalid_nav = nav[nav[nav_col] <= 0]
if len(invalid_nav) > 0:
    print(f"⚠️ {len(invalid_nav)} rows with NAV <= 0 – will be dropped")
    nav = nav[nav[nav_col] > 0]

# 7. Remove any remaining null NAV
nav = nav.dropna(subset=[nav_col])

print(f"Cleaned shape: {nav.shape}")

# Save to processed
nav.to_csv(processed_path / "02_nav_history_cleaned.csv", index=False)
print("✅ nav_history cleaned & saved")

Original shape: (46000, 3)
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
Date column: date
Cleaned shape: (46000, 3)
✅ nav_history cleaned & saved


In [4]:
trans = data['investor_transactions'].copy()
print(f"Original shape: {trans.shape}")

# --- Identify columns ---
date_col = None
for col in trans.columns:
    if 'date' in col.lower():
        date_col = col
        break

amount_col = None
for col in trans.columns:
    if 'amount' in col.lower():
        amount_col = col
        break

type_col = None
for col in trans.columns:
    if 'type' in col.lower() or 'transaction' in col.lower():
        type_col = col
        break

kyc_col = None
for col in trans.columns:
    if 'kyc' in col.lower():
        kyc_col = col
        break

# 1. Fix date formats
trans[date_col] = pd.to_datetime(trans[date_col], errors='coerce')
trans = trans.dropna(subset=[date_col])

# 2. Standardise transaction_type values (SIP / Lumpsum / Redemption)
if type_col:
    trans[type_col] = trans[type_col].astype(str).str.lower().str.strip()
    # Map common variations
    type_map = {
        'sip': 'SIP',
        'systematic': 'SIP',
        'lumpsum': 'Lumpsum',
        'lump sum': 'Lumpsum',
        'redemption': 'Redemption',
        'redeem': 'Redemption',
        'withdrawal': 'Redemption'
    }
    trans[type_col] = trans[type_col].map(lambda x: type_map.get(x, x.capitalize()))
    print(f"Unique transaction types: {trans[type_col].unique()}")

# 3. Validate amount > 0
trans = trans[trans[amount_col] > 0]

# 4. Check KYC status enum values
if kyc_col:
    trans[kyc_col] = trans[kyc_col].astype(str).str.upper().str.strip()
    valid_kyc = ['VERIFIED', 'PENDING', 'NOT_SUBMITTED', 'REJECTED']
    trans = trans[trans[kyc_col].isin(valid_kyc)]

print(f"Cleaned shape: {trans.shape}")
trans.to_csv(processed_path / "08_investor_transactions_cleaned.csv", index=False)
print("✅ investor_transactions cleaned")

Original shape: (32778, 13)
Unique transaction types: ['2024-01-01' '2024-01-02' '2024-01-03' '2024-01-04' '2024-01-05'
 '2024-01-06' '2024-01-07' '2024-01-08' '2024-01-09' '2024-01-10'
 '2024-01-11' '2024-01-12' '2024-01-13' '2024-01-14' '2024-01-15'
 '2024-01-16' '2024-01-17' '2024-01-18' '2024-01-19' '2024-01-20'
 '2024-01-21' '2024-01-22' '2024-01-23' '2024-01-24' '2024-01-25'
 '2024-01-26' '2024-01-27' '2024-01-28' '2024-01-29' '2024-01-30'
 '2024-01-31' '2024-02-01' '2024-02-02' '2024-02-03' '2024-02-04'
 '2024-02-05' '2024-02-06' '2024-02-07' '2024-02-08' '2024-02-09'
 '2024-02-10' '2024-02-11' '2024-02-12' '2024-02-13' '2024-02-14'
 '2024-02-15' '2024-02-16' '2024-02-17' '2024-02-18' '2024-02-19'
 '2024-02-20' '2024-02-21' '2024-02-22' '2024-02-23' '2024-02-24'
 '2024-02-25' '2024-02-26' '2024-02-27' '2024-02-28' '2024-02-29'
 '2024-03-01' '2024-03-02' '2024-03-03' '2024-03-04' '2024-03-05'
 '2024-03-06' '2024-03-07' '2024-03-08' '2024-03-09' '2024-03-10'
 '2024-03-11' '2024-03

In [5]:
perf = data['scheme_performance'].copy()
print(f"Original shape: {perf.shape}")

# --- Identify return columns (e.g., 1M_return, 3M_return, 1Y_return, etc.) ---
return_cols = [col for col in perf.columns if 'return' in col.lower() or 'ret' in col.lower()]

# 1. Convert return columns to numeric, coerce errors
for col in return_cols:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')

# 2. Flag anomalies: returns outside reasonable range (e.g., -100% to +200%)
for col in return_cols:
    outliers = (perf[col] < -100) | (perf[col] > 200)
    if outliers.any():
        print(f"⚠️ {outliers.sum()} outliers in {col} (outside -100 to 200)")
        # Optional: cap or remove – we'll keep for now but flag

# 3. Expense ratio check (assume column contains 'expense' or 'exp_ratio')
exp_col = None
for col in perf.columns:
    if 'expense' in col.lower() or 'exp_ratio' in col.lower():
        exp_col = col
        break

if exp_col:
    perf[exp_col] = pd.to_numeric(perf[exp_col], errors='coerce')
    invalid_exp = (perf[exp_col] < 0.1) | (perf[exp_col] > 2.5)
    print(f"Expense ratio out of range (0.1%-2.5%): {invalid_exp.sum()} rows")
    # Keep but note

perf.to_csv(processed_path / "07_scheme_performance_cleaned.csv", index=False)
print("✅ scheme_performance cleaned")

Original shape: (40, 19)
Expense ratio out of range (0.1%-2.5%): 0 rows
✅ scheme_performance cleaned


In [6]:
# aum_by_fund_house – ensure numeric AUM
aum = data['aum_by_fund_house'].copy()
aum_col = None
for col in aum.columns:
    if 'aum' in col.lower():
        aum_col = col
        break
if aum_col:
    aum[aum_col] = pd.to_numeric(aum[aum_col], errors='coerce')
aum.to_csv(processed_path / "03_aum_by_fund_house_cleaned.csv", index=False)

# monthly_sip_inflows – ensure numeric
sip = data['monthly_sip_inflows'].copy()
for col in sip.select_dtypes(include=['object']):
    sip[col] = pd.to_numeric(sip[col], errors='coerce')
sip.to_csv(processed_path / "04_monthly_sip_inflows_cleaned.csv", index=False)

# category_inflows – similar
cat = data['category_inflows'].copy()
for col in cat.select_dtypes(include=['object']):
    cat[col] = pd.to_numeric(cat[col], errors='coerce')
cat.to_csv(processed_path / "05_category_inflows_cleaned.csv", index=False)

# industry_folio_count
ind = data['industry_folio_count'].copy()
ind.to_csv(processed_path / "06_industry_folio_count_cleaned.csv", index=False)

# portfolio_holdings
port = data['portfolio_holdings'].copy()
port.to_csv(processed_path / "09_portfolio_holdings_cleaned.csv", index=False)

# benchmark_indices
bench = data['benchmark_indices'].copy()
bench.to_csv(processed_path / "10_benchmark_indices_cleaned.csv", index=False)

print("All other datasets saved to processed/")

All other datasets saved to processed/
